In [61]:
# Standard library imports
import gc
import os
import random
import warnings

# Third-party imports
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import xgboost as xgb
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from matplotlib import pyplot as plt
from pycox.datasets import metabric
from pycox.datasets import support
from pycox.models import CoxPH
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sksurv.datasets import load_breast_cancer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from tqdm.auto import tqdm
from pycox.models import DeepHitSingle
from pycox.preprocessing.label_transforms import LabTransDiscreteTime

In [62]:
# utilise gpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

Training on: cpu


In [63]:
np.random.seed(1)
n_samples = 6000
d = 10

# Linear model
X = np.random.uniform(-1, 1, (n_samples, d))
# log-risk function
f_x = X[:, 0] + 2*X[:, 1]

lambda_0 = 1.0
U = np.random.uniform(0, 1, n_samples)
# survival times: T = - log(U) / (lambda_0 * exp(f(x)))
T = -np.log(U) / (lambda_0 * np.exp(f_x))

# censoring 50% similar to paper
C = np.random.exponential(scale=np.mean(T), size=n_samples)

time = np.minimum(T, C)  # choose minimum
event = (T <= C).astype(int)

In [64]:
# # non-linear model
# np.random.seed(474)

# lambda_max = 5.0
# r = 0.5
# # (x) = log(lambda_max) * exp(- (x_0^2 + x_1^2) / (2 * r^2))
# X_nl = np.random.uniform(-1, 1, (n_samples, d))
# term = (X[:, 0]**2 + X[:, 1]**2) / (2 * r**2)
# h_x_nl = np.log(lambda_max) * np.exp(-term)

# U = np.random.uniform(0, 1, n_samples)
# # survival times: T = - log(U) / (lambda_0 * exp(h(x)))
# T_nl = -np.log(U) / (lambda_0 * np.exp(h_x_nl))

# # censoring 50% similar to paper
# C_nl = np.random.exponential(scale=np.mean(T_nl), size=n_samples)

# time_nl = np.minimum(T_nl, C_nl)  # choose minimum
# event_nl = (T_nl <= C_nl).astype(int)

In [65]:
# quadratic non-linear model
alpha = 3.0 # Controls the steepness of the "U" shape
X_nl_quad = np.random.uniform(-1, 1, (n_samples, d))

# f(x) = alpha * (x0^2 + x1^2)
# Risk is lowest at (0,0) and highest at the corners/edges
f_x_nl_quad = alpha * (X_nl_quad[:, 0]**2 + X_nl_quad[:, 1]**2)

U_quad = np.random.uniform(0, 1, n_samples)

# survival times: T = - log(U) / (lambda_0 * exp(f(x)))
T_nl_quad = -np.log(U_quad) / (lambda_0 * np.exp(f_x_nl_quad))

# censoring ~50%
C_nl_quad = np.random.exponential(scale=np.mean(T_nl_quad), size=n_samples)

time_nl_quad = np.minimum(T_nl_quad, C_nl_quad)
event_nl_quad = (T_nl_quad <= C_nl_quad).astype(int)

In [66]:
# metabric dataset
df_metabric = metabric.read_df()

X_metabric = df_metabric.drop(["duration", "event"], axis=1).values.astype("float32")
T_metabric = df_metabric["duration"].values.astype("float32")
E_metabric = df_metabric["event"].values.astype("float32")

In [67]:
# SUPPORT data
df_support = support.read_df()

X_support = df_support.drop(["duration", "event"], axis=1).values.astype("float32")
T_support = df_support["duration"].values.astype("float32")
E_support = df_support["event"].values.astype("float32")

In [68]:
df_tcga = pd.read_csv('./data/tcga_processed_lung.csv')

T_tcga = df_tcga["OS_MONTHS"].values.astype("float32")
E_tcga = df_tcga["OS_STATUS"].values.astype("float32")

# Drop the targets and colinear feature
cols_to_drop = ["OS_MONTHS", "OS_STATUS", "PANEL_MUT_BURDEN"]

X_tcga = df_tcga.drop(columns=cols_to_drop).values.astype("float32")
selector = VarianceThreshold(threshold=0.0)
X_tcga = selector.fit_transform(X_tcga)

print(f"Features (X) shape: {X_tcga.shape}")
print(f"Durations (T) shape: {T_tcga.shape}")

Features (X) shape: (935, 43)
Durations (T) shape: (935,)


In [69]:
# cox proportional hazards
def cph(X_train, T_train, E_train, X_test):
    df_train = pd.DataFrame(X_train)
    df_train["time"] = T_train
    df_train["event"] = E_train

    cph = CoxPHFitter(penalizer=0.0)  # no penalisation

    # Using a try-except here is highly recommended for standard CPH
    # It will often crash on datasets with more features than samples
    try:
        cph.fit(df_train, duration_col="time", event_col="event")

        # predictions flattened to a 1D array
        preds = cph.predict_partial_hazard(pd.DataFrame(X_test)).values.flatten()

        return preds

    except Exception as e:
        print(f"CPH failed to converge: {e}")
        return None

In [70]:
def cox_lasso(X_train, T_train, E_train, X_test):
    try:
        # scikit-survival requires a structured array for the target variable
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)],
                           dtype=[('event', 'bool'), ('time', 'float32')])

        # Initialize Lasso (l1_ratio=1.0 is pure Lasso / L1 penalty)
        lasso_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, fit_baseline_model=True)

        # fit model
        lasso_model.fit(X_train, y_train)

        # predict and flatten to ensure a clean 1D array for the bootstrap loop
        risk_scores = lasso_model.predict(X_test).flatten()

        return risk_scores

    except Exception as e:
        print(f"Cox Lasso failed to converge: {e}")
        return None

In [71]:
# random survival forest
def rsf(X_train, T_train, E_train, X_test, T_test, E_test):
  # format data for scikit-survival input
  y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[("e", bool), ("t", float)])
  y_test = np.array([(bool(e), t) for e, t in zip(E_test, T_test)], dtype=[("e", bool), ("t", float)])

  rsf = RandomSurvivalForest(n_estimators=100, min_samples_leaf=15, n_jobs=-1, random_state=0)
  rsf.fit(X_train, y_train)

  c_index = rsf.score(X_test, y_test)
  return c_index

In [72]:
def rsf_gpu(X_train, T_train, E_train, X_test):
    try:
        # format labels for xgboost cox. positive if event=1, negative if censored
        y_train = np.where(E_train == 1, T_train, -T_train)

        # random forest parameters
        params = {
            "objective": "survival:cox", # use Cox Proportional Hazards
            "eval_metric": "cox-nloglik",
            "tree_method": "hist",       # required for gpu execution
            "device": "cuda",            # utilise gpu
            "booster": "gbtree",
            "subsample": 0.8,            # Randomly sample 80% of rows per tree
            "colsample_bynode": 0.8,     # Randomly sample 80% of features per split
            "learning_rate": 1.0,        # 1.0 for random survival forest
            "num_parallel_tree": 100,    # number of trees
            "min_child_weight": 15,
        }

        # Create DMatrix for training (with labels) and testing (without labels)
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dtest = xgb.DMatrix(X_test)

        # train forest (num_boost_round=1 ensures it just builds the single forest of 100 trees)
        bst = xgb.train(params, dtrain, num_boost_round=1)

        # predict and flatten
        preds = bst.predict(dtest).flatten()

        return preds

    except Exception as e:
        print(f"RSF (XGBoost GPU) failed: {e}")
        return None

In [73]:
def deepsurv(X_train, T_train, E_train, X_test, params):
    try:
        # Ensure arrays are float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
            X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
        )

        y_tr = (T_tr, E_tr)
        y_val = (T_val, E_val)

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [params["nodes"]] * params["layers"]
        dropout = params["dropout"]
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        if params.get("optimiser", "adam") == "adam":
            optim = tt.optim.Adam(lr=params["lr"], weight_decay=params.get("l2", 0.0))
        elif params.get("optimiser") == "sgd":
            optim = tt.optim.SGD(lr=params["lr"], momentum=params.get("momentum", 0.9),
                                 weight_decay=params.get("l2", 0.0), nesterov=True)
        else:
            raise ValueError("Unknown optimiser")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]
        batch_size = params.get("batch_size", 64)

        model.fit(
            X_tr, y_tr,
            batch_size=batch_size,
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Return the flat array of log-hazard (risk) scores
        risk = model.predict_net(X_test).flatten()

        return risk

    except Exception as e:
        print(f"DeepSurv failed: {e}")
        return None

In [ ]:
def optimise_deepsurv(X_train, T_train, E_train, n_trials=30):
    X_train, T_train, E_train = X_train.astype("float32"), T_train.astype("float32"), E_train.astype("float32")

    X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
        X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
    )

    def objective(trial):
        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 16, 128, step=16)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        l2 = trial.suggest_float("l2", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        activation_name = trial.suggest_categorical("activation", ["ReLU", "SELU"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [n_nodes] * n_layers
        activation = nn.SELU if activation_name == "SELU" else nn.ReLU

        # deepsurv
        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr, weight_decay=l2)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        model.fit(
            X_tr, (T_tr, E_tr),
            batch_size=batch_size, epochs=500,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, (T_val, E_val))
        )

        risk = model.predict_net(X_val).flatten()
        c_index = concordance_index(T_val, -risk, E_val)

        # memory cleanup
        del model, net, optim
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return c_index

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [ ]:
def deephit(X_train, T_train, E_train, X_test, params):
    try:
        # Ensure float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # extract parameters
        alpha = params.get("alpha", 0.2)
        sigma = params.get("sigma", 0.1)
        num_durations = params.get("num_durations", 10)

        # discretise time for DeepHit
        labtrans = LabTransDiscreteTime(num_durations)
        # Fit the transformer on training data and transform it
        y_train_discrete = labtrans.fit_transform(T_train, E_train)

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event = train_test_split(
            X_train, y_train_discrete[0], y_train_discrete[1],
            test_size=0.2, random_state=42, stratify=y_train_discrete[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the network
        in_features = X_train.shape[1]
        out_features = labtrans.out_features # Network must output exactly the number of time bins
        num_nodes = [params.get("nodes", 32)] * params.get("layers", 2)
        dropout = params.get("dropout", 0.2)
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=params.get("lr", 1e-3), weight_decay=params.get("l2", 0.01))
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # deephit
        model = DeepHitSingle(net, optim, alpha=alpha, sigma=sigma, duration_index=labtrans.cuts, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        # train
        model.fit(
            X_tr, y_tr,
            batch_size=params.get("batch_size", 64),
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # predict
        surv_df = model.predict_surv_df(X_test)

        # compute risk
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        return risk

    except Exception as e:
        print(f"DeepHit failed: {e}")
        return None

In [76]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Sim Linear
X_train_lin, X_test_lin, T_train_lin, T_test_lin, E_train_lin, E_test_lin = train_test_split(
    X, time, event, test_size=0.2, random_state=42, stratify=event
)

# Sim Nonlinear Quadratic
X_train_quad, X_test_quad, T_train_quad, T_test_quad, E_train_quad, E_test_quad = train_test_split(
    X_nl_quad, time_nl_quad, event_nl_quad, test_size=0.2, random_state=42, stratify=event_nl_quad
)

# Metabric
X_train_met, X_test_met, T_train_met, T_test_met, E_train_met, E_test_met = train_test_split(
    X_metabric, T_metabric, E_metabric, test_size=0.2, random_state=42, stratify=E_metabric
)

# TCGA
X_train_tcga, X_test_tcga, T_train_tcga, T_test_tcga, E_train_tcga, E_test_tcga = train_test_split(
    X_tcga, T_tcga, E_tcga, test_size=0.2, random_state=42, stratify=E_tcga
)

# SUPPORT
X_train_sup, X_test_sup, T_train_sup, T_test_sup, E_train_sup, E_test_sup = train_test_split(
    X_support, T_support, E_support, test_size=0.2, random_state=42, stratify=E_support
)

In [77]:
n_trials = 20

best_linear = optimise_deepsurv(X_train_lin, T_train_lin, E_train_lin, n_trials=n_trials)
params_linear = best_linear.copy()

best_nonlinear_quad = optimise_deepsurv(X_train_quad, T_train_quad, E_train_quad, n_trials=n_trials)
params_nonlinear_quad = best_nonlinear_quad.copy()


# Metabric
scaler_met = StandardScaler()
X_train_met_scaled = scaler_met.fit_transform(X_train_met)
best_met_params = optimise_deepsurv(X_train_met_scaled, T_train_met, E_train_met, n_trials=n_trials)
params_metabric = best_met_params.copy()

# TCGA
scaler_tcga = StandardScaler()
X_train_tcga_scaled = scaler_tcga.fit_transform(X_train_tcga)
best_tcga_params = optimise_deepsurv(X_train_tcga_scaled, T_train_tcga, E_train_tcga, n_trials=n_trials)
params_tcga = best_tcga_params.copy()

# SUPPORT
scaler_sup = StandardScaler()
X_train_sup_scaled = scaler_sup.fit_transform(X_train_sup)
best_sup_params = optimise_deepsurv(X_train_sup_scaled, T_train_sup, E_train_sup, n_trials=n_trials)
params_support = best_sup_params.copy()

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [78]:
# Print all optimal hyperparameters
print(f"params_linear = {params_linear}")
# print(f"params_nonlinear = {params_nonlinear}")
print(f"params_nonlinear_quad = {params_nonlinear_quad}")
print(f"params_metabric = {params_metabric}")
print(f"params_tcga = {params_tcga}")
print(f"params_support = {params_support}")

params_linear = {'nodes': 96, 'layers': 2, 'dropout': 0.5025834440490632, 'lr': 0.008659003206637988, 'l2': 0.0006316091506704555, 'batch_size': 32, 'activation': 'SELU', 'batch_norm': False}
params_nonlinear_quad = {'nodes': 48, 'layers': 1, 'dropout': 0.25494072502727716, 'lr': 0.0030534378821209926, 'l2': 0.0012491059109425975, 'batch_size': 128, 'activation': 'SELU', 'batch_norm': True}
params_metabric = {'nodes': 128, 'layers': 4, 'dropout': 0.20932469451038052, 'lr': 0.0017893749707028816, 'l2': 0.0009865754363454753, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': False}
params_tcga = {'nodes': 64, 'layers': 3, 'dropout': 0.4932310428335782, 'lr': 0.00931940253621531, 'l2': 0.01698587948715676, 'batch_size': 32, 'activation': 'ReLU', 'batch_norm': False}
params_support = {'nodes': 96, 'layers': 2, 'dropout': 0.1005411431182513, 'lr': 0.0021003928884391868, 'l2': 0.010731116098513342, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': True}


In [79]:
# params_linear = {'nodes': 64, 'layers': 3, 'dropout': 0.1524398771923285, 'lr': 0.0042284170430759915, 'l2': 0.00240932336631195, 'batch_size': 64, 'activation': 'SELU', 'batch_norm': True}
# params_nonlinear_quad = {'nodes': 80, 'layers': 1, 'dropout': 0.29863460440350675, 'lr': 0.0003157288902407482, 'l2': 0.0017804504564193338, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': False}
# params_metabric = {'nodes': 96, 'layers': 2, 'dropout': 0.40682554444661706, 'lr': 0.0055054833749999515, 'l2': 0.07682436313155583, 'batch_size': 128, 'activation': 'SELU', 'batch_norm': False}
# params_tcga = {'nodes': 96, 'layers': 2, 'dropout': 0.5123373951450122, 'lr': 0.0065266165690655905, 'l2': 0.007522708444977341, 'batch_size': 128, 'activation': 'SELU', 'batch_norm': False}
# params_support = {'nodes': 64, 'layers': 4, 'dropout': 0.11832330719452677, 'lr': 0.002455139999819192, 'l2': 0.004253793680492086, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': True}

In [ ]:
def optimise_deephit(X_train, T_train, E_train, n_trials=15):
    X_train = X_train.astype("float32")
    T_train = T_train.astype("float32")
    E_train = E_train.astype("float32")

    def objective(trial):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 32, 64, step=32)
        n_layers = trial.suggest_int("layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.2, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128])

        alpha = trial.suggest_float("alpha", 0.0, 1.0)
        sigma = trial.suggest_float("sigma", 0.1, 1.0)
        num_durations = trial.suggest_categorical("num_durations", [10, 20])

        # deephit
        labtrans = LabTransDiscreteTime(num_durations)
        y_train_discrete = labtrans.fit_transform(T_train, E_train)

        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event, T_tr_orig, T_val_orig, E_tr_orig, E_val_orig = train_test_split(
            X_train, y_train_discrete[0], y_train_discrete[1], T_train, E_train,
            test_size=0.2, random_state=42, stratify=y_train_discrete[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the network
        in_features = X_tr.shape[1]
        out_features = labtrans.out_features
        num_nodes_list = [n_nodes] * n_layers

        net = tt.practical.MLPVanilla(
            in_features, num_nodes_list, out_features,
            batch_norm=True, dropout=dropout,
            activation=nn.ReLU, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        model = DeepHitSingle(net, optim, alpha=alpha, sigma=sigma, duration_index=labtrans.cuts, device=device)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)] # Lowered patience for speed/memory

        # train the model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size, epochs=200, # Lowered max epochs
            callbacks=callbacks, verbose=False,
            val_data=(X_val, y_val)
        )

        # compute risk
        surv_df = model.predict_surv_df(X_val)
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        c_index = concordance_index(T_val_orig, risk, E_val_orig)

        # memory cleanup
        del model, net, optim, surv_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return c_index

    # Run the study
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [81]:
# Lowered n_trials for DeepHit to prevent Out-Of-Memory kernel crashes
n_trials_dh = 20

# Sim Linear
best_linear_dh = optimise_deephit(X_train_lin, T_train_lin, E_train_lin, n_trials=n_trials_dh)
params_linear_dh = best_linear_dh.copy()

# Sim Nonlinear Quadratic
best_nonlinear_quad_dh = optimise_deephit(X_train_quad, T_train_quad, E_train_quad, n_trials=n_trials_dh)
params_nonlinear_quad_dh = best_nonlinear_quad_dh.copy()


# Metabric
scaler_met = StandardScaler()
X_train_met_scaled = scaler_met.fit_transform(X_train_met)
best_met_params_dh = optimise_deephit(X_train_met_scaled, T_train_met, E_train_met, n_trials=n_trials_dh)
params_metabric_dh = best_met_params_dh.copy()

# TCGA
scaler_tcga = StandardScaler()
X_train_tcga_scaled = scaler_tcga.fit_transform(X_train_tcga)
best_tcga_params_dh = optimise_deephit(X_train_tcga_scaled, T_train_tcga, E_train_tcga, n_trials=n_trials_dh)
params_tcga_dh = best_tcga_params_dh.copy()

# SUPPORT
scaler_sup = StandardScaler()
X_train_sup_scaled = scaler_sup.fit_transform(X_train_sup)
best_sup_params_dh = optimise_deephit(X_train_sup_scaled, T_train_sup, E_train_sup, n_trials=n_trials_dh)
params_support_dh = best_sup_params_dh.copy()

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [82]:
# Print all optimal hyperparameters
print(f"params_linear_dh = {params_linear_dh}")
# print(f"params_nonlinear = {params_nonlinear}")
print(f"params_nonlinear_quad_dh = {params_nonlinear_quad_dh}")
print(f"params_metabric_dh = {params_metabric_dh}")
print(f"params_tcga_dh = {params_tcga_dh}")
print(f"params_support_dh = {params_support_dh}")

params_linear_dh = {'nodes': 32, 'layers': 2, 'dropout': 0.24267934374598685, 'lr': 0.0001177718189191776, 'batch_size': 128, 'alpha': 0.608471812022899, 'sigma': 0.3244454049006224, 'num_durations': 10}
params_nonlinear_quad_dh = {'nodes': 32, 'layers': 1, 'dropout': 0.2954951555489345, 'lr': 0.00010435141647509268, 'batch_size': 64, 'alpha': 0.9967411772634021, 'sigma': 0.868495294715536, 'num_durations': 20}
params_metabric_dh = {'nodes': 32, 'layers': 1, 'dropout': 0.35316225919065347, 'lr': 0.00024324444881810526, 'batch_size': 128, 'alpha': 0.513700855287133, 'sigma': 0.11800108272028009, 'num_durations': 20}
params_tcga_dh = {'nodes': 64, 'layers': 1, 'dropout': 0.3019164789479899, 'lr': 0.0002077476415210216, 'batch_size': 64, 'alpha': 0.34774262765996233, 'sigma': 0.6798841496159822, 'num_durations': 10}
params_support_dh = {'nodes': 32, 'layers': 2, 'dropout': 0.2310750058847516, 'lr': 0.0014018746394226919, 'batch_size': 64, 'alpha': 0.02478430133319187, 'sigma': 0.112400421

In [83]:
# params_linear_dh = {'nodes': 32, 'layers': 2, 'dropout': 0.22937981467625856, 'lr': 0.00033416036208661743, 'batch_size': 128, 'alpha': 0.24531306225918015, 'sigma': 0.9427460024939718, 'num_durations': 10}
# params_nonlinear_quad_dh = {'nodes': 32, 'layers': 3, 'dropout': 0.5953575872008865, 'lr': 0.00011708866519224654, 'batch_size': 64, 'alpha': 0.44126113357661456, 'sigma': 0.7942692908443046, 'num_durations': 10}
# params_metabric_dh = {'nodes': 32, 'layers': 3, 'dropout': 0.5541766066031479, 'lr': 0.00019026569115133615, 'batch_size': 128, 'alpha': 0.5977397895687814, 'sigma': 0.30835546690412413, 'num_durations': 20}
# params_tcga_dh = {'nodes': 32, 'layers': 2, 'dropout': 0.29056644213231625, 'lr': 0.002437169764507907, 'batch_size': 64, 'alpha': 0.5671558694808683, 'sigma': 0.41061959465242026, 'num_durations': 20}
# params_support_dh = {'nodes': 64, 'layers': 1, 'dropout': 0.5920911102752608, 'lr': 0.0009301107552843405, 'batch_size': 64, 'alpha': 0.5042890877664027, 'sigma': 0.30436895887926285, 'num_durations': 20}

In [ ]:
def evaluate_dataset(X_train, T_train, E_train,
                     X_test, T_test, E_test,
                     dataset_name, params_ds, params_dh,
                     n_bootstraps=1000):

    # Scale features strictly on the training data to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Convert targets to numpy arrays for easier indexing during bootstrapping
    T_test = np.array(T_test)
    E_test = np.array(E_test)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # train models
        risk_cph = cph(X_train_scaled, T_train, E_train, X_test_scaled)
        risk_lasso = cox_lasso(X_train_scaled, T_train, E_train, X_test_scaled)
        risk_ds = deepsurv(X_train_scaled, T_train, E_train, X_test_scaled, params_ds)
        risk_rsf = rsf_gpu(X_train_scaled, T_train, E_train, X_test_scaled)
        risk_dh = deephit(X_train_scaled, T_train, E_train, X_test_scaled, params_dh)

    boot_cph, boot_lasso, boot_ds, boot_rsf, boot_dh = [], [], [], [], []

    # bootstrap to comput c-index
    for i in tqdm(range(n_bootstraps), desc=f"Bootstrapping {dataset_name}", leave=False):
        # Resample the test set indices with replacement
        indices = resample(np.arange(len(X_test_scaled)), replace=True, random_state=i)

        T_boot = T_test[indices]
        E_boot = E_test[indices]

        if np.sum(E_boot) == 0:
            continue

        # Calculate C-index for each model on this specific resampled test set
        try:
            if risk_cph is not None:
                boot_cph.append(concordance_index(T_boot, -risk_cph[indices], E_boot))
            if risk_lasso is not None:
                boot_lasso.append(concordance_index(T_boot, -risk_lasso[indices], E_boot))
            if risk_ds is not None:
                boot_ds.append(concordance_index(T_boot, -risk_ds[indices], E_boot))
            if risk_rsf is not None:
                boot_rsf.append(concordance_index(T_boot, -risk_rsf[indices], E_boot))
            if risk_dh is not None:
                boot_dh.append(concordance_index(T_boot, -risk_dh[indices], E_boot)) # <-- Added DeepHit
        except Exception:
            continue

    # Helper function to extract Mean and 95% CI from the bootstrap lists
    def get_metrics(boot_list):
        if not boot_list:
            return np.nan, (np.nan, np.nan)
        return np.mean(boot_list), (round(np.percentile(boot_list, 2.5), 3), round(np.percentile(boot_list, 97.5), 3))

    cph_mean, cph_ci = get_metrics(boot_cph)
    lasso_mean, lasso_ci = get_metrics(boot_lasso)
    ds_mean, ds_ci = get_metrics(boot_ds)
    rsf_mean, rsf_ci = get_metrics(boot_rsf)
    dh_mean, dh_ci = get_metrics(boot_dh)
    return {
        "Dataset": dataset_name,
        "CPH (Mean)": cph_mean,
        "CPH (95% CI)": cph_ci,
        "Cox Lasso (Mean)": lasso_mean,
        "Cox Lasso (95% CI)": lasso_ci,
        "DeepSurv (Mean)": ds_mean,
        "DeepSurv (95% CI)": ds_ci,
        "DeepHit (Mean)": dh_mean,
        "DeepHit (95% CI)": dh_ci,
        "RSF (Mean)": rsf_mean,
        "RSF (95% CI)": rsf_ci
    }

In [85]:
# 1,000 is the standard for calculating reliable 95% Confidence Intervals via bootstrapping
n_bootstraps = 10
final_results = []

# Sim Linear
final_results.append(evaluate_dataset(
    X_train_lin, T_train_lin, E_train_lin,
    X_test_lin, T_test_lin, E_test_lin,
    "Sim Linear", params_linear, params_linear_dh, n_bootstraps
))

# Sim Nonlinear Quadratic
final_results.append(evaluate_dataset(
    X_train_quad, T_train_quad, E_train_quad,
    X_test_quad, T_test_quad, E_test_quad,
    "Sim Nonlinear Quadratic", params_nonlinear_quad, params_nonlinear_quad_dh, n_bootstraps
))

# Metabric
final_results.append(evaluate_dataset(
    X_train_met, T_train_met, E_train_met,
    X_test_met, T_test_met, E_test_met,
    "Metabric", params_metabric, params_metabric_dh, n_bootstraps
))

# TCGA
final_results.append(evaluate_dataset(
    X_train_tcga, T_train_tcga, E_train_tcga,
    X_test_tcga, T_test_tcga, E_test_tcga,
    "TCGA", params_tcga, params_tcga_dh, n_bootstraps
))

# SUPPORT
final_results.append(evaluate_dataset(
    X_train_sup, T_train_sup, E_train_sup,
    X_test_sup, T_test_sup, E_test_sup,
    "SUPPORT", params_support, params_support_dh, n_bootstraps
))

df_final = pd.DataFrame(final_results)

# Print for your console
print(df_final.to_string(index=False))

Bootstrapping Sim Linear:   0%|          | 0/10 [00:00<?, ?it/s]

Bootstrapping Sim Nonlinear Quadratic:   0%|          | 0/10 [00:00<?, ?it/s]

Bootstrapping Metabric:   0%|          | 0/10 [00:00<?, ?it/s]

Bootstrapping TCGA:   0%|          | 0/10 [00:00<?, ?it/s]

Bootstrapping SUPPORT:   0%|          | 0/10 [00:00<?, ?it/s]

                Dataset  CPH (Mean)   CPH (95% CI)  Cox Lasso (Mean) Cox Lasso (95% CI)  DeepSurv (Mean) DeepSurv (95% CI)  DeepHit (Mean) DeepHit (95% CI)  RSF (Mean)   RSF (95% CI)
             Sim Linear    0.783437 (0.773, 0.795)          0.783678     (0.773, 0.795)         0.781429    (0.773, 0.792)        0.775370   (0.764, 0.788)    0.779229 (0.769, 0.791)
Sim Nonlinear Quadratic    0.521421 (0.499, 0.546)          0.521396     (0.499, 0.546)         0.772883      (0.76, 0.79)        0.764299    (0.75, 0.779)    0.768727 (0.756, 0.784)
               Metabric    0.616196  (0.574, 0.65)          0.615591     (0.574, 0.649)         0.618930    (0.588, 0.644)        0.589478   (0.548, 0.638)    0.623271 (0.592, 0.656)
                   TCGA    0.619452 (0.524, 0.687)          0.620464     (0.526, 0.686)         0.625890    (0.533, 0.678)        0.624265   (0.557, 0.696)    0.712288 (0.656, 0.767)
                SUPPORT    0.570416 (0.561, 0.586)          0.570326     (0.561, 0.58